# Decorator

In [43]:
from collections.abc import Callable
from typing import Any

## Intro

There is a `foo` function that accepts a string and returns its modified version. Imagine that this function is crucial and neither its signature nor body cannot be changed. In certain cases the default formatted message must be different, so the task is "to enclose the original message into brackets". Hm, easy-peasy, just define another function that reuses the target (`foo`) function and call it whenever it is should be.

In [26]:
def foo(s: str) -> str:
    """Returns a fancy string.

    Args:
        s (str): original string

    Returns:
        str
    """
    return f"string is {s!r}"


def add_brackets_v1(s: str) -> str:
    return f"[{foo(s)}]"


s = 'hello'
print(f"original: {foo(s)}")
print(f"modified: {add_brackets(s)}")

original: string is 'hello'
modified: [hello]


An ugly solution for `add_brackets` depends on `foo`, so it is just a function that can reuse only the `foo` function and that is all. Ok, my bad, `add_brackets` can be less coupled with `foo` and be used with any string.

In [27]:
def add_brackets_v2(s: str) -> str:
    return f"[{s}]"


s = 'hello'
print(f"original: {foo(s)}")
print(f"modified: {add_brackets_v2(foo(s))}")

original: string is 'hello'
modified: [string is 'hello']


Later, the new wish combinations are asked, so now to maintain four cases:
1. a string in brackets (original -> "[s]")
2. a string in parentheses ("(s)")
3. case 1 and then case 2 ("([s])")
4. case 2 and then case 1 ("[(s)]")

Damn, revise the `add_brackets` and define `add_parentheses` and defi...no, just using their composition!

In [28]:
def add_brackets(s: str) -> str:
    return f"[{s}]"


def add_parentheses(s: str) -> str:
    return f"({s})"


s = 'hello'
print(f"foo(s): {foo(s)}")
print(f"[foo(s)]: {add_brackets(foo(s))}")
print(f"(foo(s)): {add_parentheses(foo(s))}")
print(f"[(foo(s))]: {add_brackets(add_parentheses(foo(s)))}")
print(f"([foo(s)]): {add_parentheses(add_brackets(foo(s)))}")

foo(s): string is 'hello'
[foo(s)]: [string is 'hello']
(foo(s)): (string is 'hello')
[(foo(s))]: [(string is 'hello')]
([foo(s)]): ([string is 'hello'])


It may seem slick, yet 1) it is static and forms 2) a string of calls which are 3) functional but verbose in a way. Is there an option to reduce this chain of invocations to just one call? In short, yes and wrappers can deal with this task just fine. Actually for this case exists the Decorator, or Wrapper, pattern.

In [29]:
def bracketize(func: Callable) -> Callable:
    # `func` is local within `bracketize` decorator (!)
    # `func` is enclosed (!) for `wrapper`
    def wrapper(*args, **kwargs):
        return f"[{func(*args, **kwargs)}]"
    # return a function that can accept any parametres and produce anything
    # every time `wrapper` is called, `func` is also called (closure)
    return wrapper


def parenthesize(func: Callable) -> Callable:
    # another decorating function `parenthesize` that suits
    # `_` is a valid name for an actual wrapping function
    # because, again, the decorator pattern is also known as the wrapper pattern
    def _(*args, **kwargs):
        return f"({func(*args, **kwargs)})"
    return _


bra = bracketize(foo)
# bra references to the result of `bracketize`
# which is its inner `wrapper` function
# that aceepts *params, **kwparams
# that are given to the wrapped `foo` function
par = parenthesize(foo)

s1 = "tada"
print(s1)

print(bra(s1))  # case 1 - done
print(par(s1))  # case 2 - done

bra_par = bracketize(par)
# par is callable, so `bracketize` can works with it
# it is the same as `bracketize(parenthesize(foo))`
par_bra = parenthesize(bra)

print(bra_par(s1))
print(par_bra(s1))

tada
[string is 'tada']
(string is 'tada')
[(string is 'tada')]
([string is 'tada'])


In Python, callable objects (functions, but not only) can be decorated with `@decorator` syntax. It follows Python's idiomacy when callable objects must be decorated permanently which means statically.

In [35]:
@parenthesize
# since the inner wrapper is `wrapper(*args, **kwargs)`
# the following function can be called without restrictions
@bracketize
# the order matters
def goo(a: int, b: int) -> float:
    return (a + b) ** .5
# equivalent to `goo = parenthesize(bracketize(goo))`

@bracketize
@parenthesize
def hoo(*args) -> int:
    return len(args)
# equivalent to `hoo = bracketize(parenthesize(hoo))`

print(goo(5, 2))
print(hoo(1, 4, 8))

([2.6457513110645907])
[(3)]


Well, I would like to have the result of `goo` rounded to the 2nd digit after the decimal point. Easy, another decorator.

In [36]:
def round_two(cb: Callable) -> Callable:
    def _deco(*args, **kwargs):
        return float(cb(*args, **kwargs))
    return _deco


@parenthesize
@bracketize
@round_two
def goo_v2(a: int, b: int) -> float:
    # not reusing goo on purpose -> consider this standalone
    return round((a + b) ** .5, 2)

# goo_v2 = parenthesize(bracketize(round_two(goo_v2)))


print(goo(5, 2))
print(goo_v2(5, 2))

([2.6457513110645907])
([2.65])


## Decorators with extra parametres

Decorators consume callable objects. What if passing any other parameters that tune a decorator and change its state/behaviour. The following example will definitely fail.

In [49]:
# try to remove the default value for `suffix` parametre...aha
def suffux(func: Callable, suffix: str = "") -> Callable:
    def _wrapper(*args, **kwargs):
        return f"{func(*args, **kwargs)} -> {suffix}"
    return _wrapper


def greeter(obj: Any = None) -> str:
    return f"Greetings, {obj}!"


greeter1 = suffux(greeter, "laddie or lassie")
print(greeter1())  # it works, `suffix` parametre is enclosed
print(greeter1("string"))  # prints the expected result


@suffux
def greeter2(obj: Any) -> str:
    return f"Hello, {obj}"


print(greeter2(21))
# and how to change the suffix for the greeter2? :)
# with `greeter1 = suffux(greeter, "laddie or lassie")` it worked
# with greeter2 it is misused!

Greetings, None! -> laddie or lassie
Greetings, string! -> laddie or lassie
Hello, 21 -> 


The situation goes bad if the default argument is not envisaged.

In [55]:
# dammit!
def prefix(func: Callable, prefix: str) -> Callable:
    def wrapper(*args, **kwargs):
        result = func(*args, **kwargs)
        return f"{prefix} -> {result}"
    return wrapper


# No way anymore
# @prefix
# def answer() -> int:
#     return 42


# Nice try...not at all :)
# @prefix(prefix="no way")
# def answer() -> int:
#     return 42

# The same as `answer = prefix(prefix="no way")`...shit!
# `func` parametre is required!
# `@prefix(prefix="no way")` is calling the prefix (decorating) function
# and this function requires a Callable at his time which is not the `answer` function

What to do?! Hints before the following solution:
1. you can call a decorator function with an argument -> `@deco(param)`;
2. this `param` can be an enclosed variable in the scope of the `deco` function;
3. calling `@deco(param)` can return a function that expects a function that wraps ...
4. ...
5. maybe PROFIT!

In [ ]:
def deco(prefix: str = "prefix", suffix: str = "suffix") -> Callable:
    def decorator(func: Callable) -> Callable:
        def _wrapper(*args, **kwargs):
            result = func(*args, **kwargs)
            return f"{prefix} > {result} < {suffix}"
        return _wrapper
    return decorator
    # `deco("p", "s")` returns `decorator`
    # that can consume a function,
    # i.e., `decorator(func)` returns a `_wrapper`
    # that is called as `_wrapper(*args, **kwargs)`
    # that invokes the wrapped `func`


@deco("H", "T")
def half_answer() -> int:
    return 21
# equivalent form:
# half_answer = deco("H", "T")(half_answer)

print(half_answer())

H > 21 < T


A "classic" example is an adder or a multiplicator -> one decorator, many functions.

In [66]:
def multiplier(coef: float = 1) -> Callable:
    def _deco(func: Callable) -> Callable:
        def _(*args, **kwargs):
            return coef * func(*args, **kwargs)
        return _
    return _deco


@multiplier()
def twelve() -> int:
    return 12


@multiplier(-2)
def twenty_one() -> int:
    return 12


@multiplier(.5)
def fourty_two() -> int:
    return 42


print(twelve())  # yes
print(twenty_one())  # not exactly -> -42
print(fourty_two())  # twisted -> 21.0


# even like this
invariant = multiplier(-.5)(multiplier(-2)(lambda: 10))
print(invariant())

12
-24
21.0
10.0


Powerful trick to be kept in head...pockets also will do.